In [ ]:
%%capture
import os
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
# import seaborn as sns
import scipy.stats as stats
from datetime import datetime
from django_pandas.io import read_frame
from pathlib import Path
from dj_notebook import activate
# pd.options.mode.copy_on_write = True
# pd.options.mode.chained_assignment = "raise"
env_file = os.environ["INTECOMM_ENV"]
documents_folder = os.environ["INTECOMM_DOCUMENTS_FOLDER"]
plus = activate(dotenv_file=env_file)

report_folder = Path(documents_folder)

In [ ]:
from edc_appointment.models import Appointment, AppointmentType

In [ ]:
# get appointment df
df_appt = read_frame(Appointment.objects.all(), verbose=False)
df1 = read_frame(AppointmentType.objects.values("id", "name").all())
df1 = df1.rename(columns={"id": "appt_type"})
df_appt = df_appt.merge(df1, on="appt_type", how="left")
df_appt = df_appt.drop(columns=["appt_type"])
df_appt = df_appt.rename(columns={"name": "appt_type"})

In [ ]:
def get_tab(df1):
    """Return a crosstab dataframe"""
    tab = pd.crosstab(index=df1["schedule_name"], columns=df1["appt_type"])
    tab_prop = tab.div(tab.sum(axis=1), axis=0).round(4) * 100
    tab = pd.concat([tab, tab_prop], axis=1)
    tab = tab.reset_index()
    tab.columns = ["arm", "facility","community", "home", "telephone", "facility%","community%", "home%", "telephone%"]
    tab = tab.replace("comm_schedule", "community_arm")
    tab = tab.replace("inte_schedule", "facility_arm")
    tab.reset_index()
    return tab

In [ ]:
def get_table(df2):
    """Return the same df with a `total` row"""
    tbl = df2.copy()
    tot = tbl["facility"].sum() + tbl["community"].sum() + tbl["home"].sum() + tbl["telephone"].sum()
    tbl.loc[2] = [
        "total", tab["facility"].sum(), tab["community"].sum(), tab["home"].sum(), tab["telephone"].sum(),
        tab["facility"].sum()/tot*100, tab["community"].sum()/tot*100, tab["home"].sum()/tot*100, tab["telephone"].sum()/tot*100]
    tbl = tbl.round(2)
    return tbl

In [ ]:
# conditions
cond_not1120 = (df_appt.visit_code!="1120")
cond_1120 = (df_appt.visit_code=="1120")
cond_status = ((df_appt.appt_status == "done") | (df_appt.appt_status == "incomplete") | (df_appt.appt_status == "in_progress"))
cond_ug = df_appt["site"]<200
cond_tz = df_appt["site"]>=200


In [ ]:
# uganda

df = df_appt[cond_not1120 & cond_status & cond_ug].copy()
df = df.reset_index(drop=True)
tab = get_tab(df)
tbl = get_table(tab)
tbl

In [ ]:
# tanzania
df = df_appt[cond_not1120 & cond_status & cond_tz].copy()
df = df.reset_index(drop=True)
tab = get_tab(df)
tbl = get_table(tab)
tbl

In [ ]:
# both
df = df_appt[cond_not1120 & cond_status].copy()
df = df.reset_index(drop=True)
tab = get_tab(df)
tbl = get_table(tab)
tbl

In [ ]:
# both looking only at the 1120 visit
df = df_appt[cond_1120 & cond_status].copy()
df = df.reset_index(drop=True)
tab = pd.crosstab(index=df["schedule_name"], columns=df["appt_type"])
tab_prop = tab.div(tab.sum(axis=1), axis=0).round(4) * 100
tab = pd.concat([tab, tab_prop], axis=1)
tab = tab.reset_index()
tab.columns = ["arm", "facility","community", "facility%","community%"]
tab = tab.replace("comm_schedule", "community_arm")
tab = tab.replace("inte_schedule", "facility_arm")
tab.reset_index()
tbl = tab.copy()
tot = tbl["facility"].sum() + tbl["community"].sum()
tbl.loc[2] = [
    "total", tab["facility"].sum(), tab["community"].sum(),
    tab["facility"].sum()/tot*100, tab["community"].sum()/tot*100]
tbl = tbl.round(2)
tbl

In [ ]:

df_problem = df_appt[cond_1120 & cond_status & (df_appt.schedule_name == "comm_schedule") & (df_appt.appt_type=="community")][["subject_identifier", "site", "visit_code", "visit_code_sequence"]]

In [ ]:
df_problem.site.value_counts().to_frame().sort_values(by=["site"]).reset_index()

In [ ]:
df_problem = df_problem.sort_values(by=["subject_identifier", "visit_code", "visit_code_sequence"]).reset_index(drop=True)

In [ ]:
# df_problem.to_csv(report_folder / "appts_1120_not_at_facility.csv")

In [ ]:
df_problem = df_problem.rename(columns={"site": "site_id"})
df_problem